# Model 1: Exploratory Data Analysis & Topic-Label Feasibility

**Project:** AI-Powered Study Assistant for Engineering Students (AI252TA — SDG 4)  
**Workstream:** Member 1 — Question-Topic Classification (Model 1)  
**Candidate Dataset:** `Omarrran/StackPulse_778K_QnA_Code_dataset` (`full` configuration)  
**Status:** Phase 1 Exploratory Profiling (Bounded Stream Sample)  

> **Notebook Purpose & Resource Policy:**  
> Per project guidelines (`notebooks/README.md`), this notebook is reserved for exploratory profiling and hypothesis validation.  
> To prevent out-of-memory errors and excessive network overhead on free/constrained environments, it streams a bounded sample (2,500 rows) directly from the Hugging Face JSONL source rather than materializing the full 778K-row dataset.  
> All class distribution statistics produced here are **exploratory estimates**, not final training distributions.

In [ ]:
import json
import urllib.request
import time
from collections import Counter
import pandas as pd
import numpy as np
import yaml

# Set display options for clean inspection
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)
print('Environment and dependencies initialized successfully.')

## 1. Bounded Streaming Data Ingestion

We stream 2,500 rows from `data/stackoverflow_778k_full.jsonl` on Hugging Face to profile schema and tag distributions without local dataset materialization.

In [ ]:
HF_JSONL_URL = 'https://huggingface.co/datasets/Omarrran/StackPulse_778K_QnA_Code_dataset/resolve/main/data/stackoverflow_778k_full.jsonl'
SAMPLE_TARGET = 2500

print(f'Streaming {SAMPLE_TARGET} rows from Hugging Face...')
start_time = time.time()
req = urllib.request.Request(HF_JSONL_URL, headers={'User-Agent': 'Mozilla/5.0'})

rows = []
with urllib.request.urlopen(req, timeout=30) as resp:
    for line in resp:
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line.decode('utf-8')))
            if len(rows) >= SAMPLE_TARGET:
                break
        except Exception:
            continue

df = pd.DataFrame(rows)
load_time = time.time() - start_time
print(f'Successfully streamed {len(df)} rows in {load_time:.2f} seconds.')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB')
print(f'Exposed columns ({len(df.columns)}): {list(df.columns)}')

## 2. Data Quality & Schema Integrity

We verify missing value rates, duplicate question IDs, and field type consistency.

In [ ]:
# 2.1 Null Value Check
null_counts = df.isnull().sum()
total_nulls = null_counts.sum()
print(f'Total null values across all 24 columns: {total_nulls}')
if total_nulls > 0:
    print(null_counts[null_counts > 0])
else:
    print('Data Quality: Zero missing/null fields across all sampled columns.')

# 2.2 Primary Key & Duplicate ID Check
dup_ids = df['id'].duplicated().sum()
print(f'Duplicate Question IDs: {dup_ids} / {len(df)} ({dup_ids / len(df) * 100:.2f}%)')

## 3. Text Field Availability & Length Distribution

Model 1 will take cleaned question text (`title` + `body_text`) as input. We analyze character and word length distributions.

In [ ]:
text_summary = []
for col in ['title', 'body_text', 'question_body']:
    char_lens = df[col].astype(str).str.len()
    word_lens = df[col].astype(str).str.split().str.len()
    text_summary.append({
        'Field': col,
        'Mean Chars': round(char_lens.mean(), 1),
        'Median Chars': round(char_lens.median(), 1),
        'P95 Chars': round(char_lens.quantile(0.95), 1),
        'Max Chars': char_lens.max(),
        'Mean Words': round(word_lens.mean(), 1),
        'Median Words': round(word_lens.median(), 1),
        'P95 Words': round(word_lens.quantile(0.95), 1),
        'Zero Length Rows': (char_lens == 0).sum()
    })

text_stats_df = pd.DataFrame(text_summary)
print(text_stats_df.to_string(index=False))

# Check code block prevalence
print(f'\nQuestions containing code blocks: {df["has_code"].sum()} / {len(df)} ({df["has_code"].mean() * 100:.1f}%)')
print(f'Mean code blocks per question: {df["code_block_cnt"].mean():.2f}')

## 4. Tag Delimitation & Frequency Profiling

We verify the tag delimiter (`|`), inspect the tag count distribution per question, and analyze high-frequency tags.

In [ ]:
# Parse pipe-delimited tags
df['tag_list'] = df['tags'].apply(lambda t: t.split('|') if isinstance(t, str) and t.strip() else [])
df['num_tags'] = df['tag_list'].apply(len)

print('Distribution of tag count per question:')
tag_dist = df['num_tags'].value_counts().sort_index()
for k, v in tag_dist.items():
    print(f'  {k} tag(s): {v:4d} ({v / len(df) * 100:5.1f}%)')

all_tags = [t for tags in df['tag_list'] for t in tags]
tag_counts = Counter(all_tags)
print(f'\nTotal tag instances: {len(all_tags)}')
print(f'Unique distinct tags: {len(tag_counts)}')
print(f'Mean tags per question: {df["num_tags"].mean():.2f} (Median: {df["num_tags"].median():.1f})')

print('\nTop 25 Most Frequent Tags in Sample:')
for tag, cnt in tag_counts.most_common(25):
    print(f'  {tag:<22} : {cnt:4d} ({cnt / len(df) * 100:5.1f}%)')

## 5. Topic Taxonomy Mapping & Feasibility Assessment

We evaluate the ten provisional topic categories from `configs/topic_map.yaml` against observed tags.  
To prevent label corruption, we use **defensible anchor mappings** for domain-specific tags and deliberately exclude standalone language names (`python`, `javascript`, `c#`, `java`, `php`) that do not denote a specific functional topic.

In [ ]:
with open('../configs/topic_map.yaml', 'r', encoding='utf-8') as f:
    tmap = yaml.safe_load(f)

provisional_topics = [t['id'] for t in tmap.get('topics', [])]
print(f'Loaded {len(provisional_topics)} provisional topics from configs/topic_map.yaml:')
for tid in provisional_topics:
    print(f'  - {tid}')

# Defensible, non-hallucinated anchor mapping based strictly on verified domain-specific tags
anchor_tag_map = {
    # 1. Algorithms & Data Structures
    'algorithms': 'algorithms_and_data_structures',
    'data-structures': 'algorithms_and_data_structures',
    'sorting': 'algorithms_and_data_structures',
    'recursion': 'algorithms_and_data_structures',
    'tree': 'algorithms_and_data_structures',
    'binary-tree': 'algorithms_and_data_structures',
    'graph': 'algorithms_and_data_structures',
    'time-complexity': 'algorithms_and_data_structures',
    'dynamic-programming': 'algorithms_and_data_structures',
    # 2. Web Frontend
    'html': 'web_frontend',
    'css': 'web_frontend',
    'reactjs': 'web_frontend',
    'angularjs': 'web_frontend',
    'angular': 'web_frontend',
    'vue.js': 'web_frontend',
    'frontend': 'web_frontend',
    'dom': 'web_frontend',
    'bootstrap': 'web_frontend',
    'jquery': 'web_frontend',
    # 3. Web Backend & APIs
    'node.js': 'web_backend_and_apis',
    'django': 'web_backend_and_apis',
    'flask': 'web_backend_and_apis',
    'fastapi': 'web_backend_and_apis',
    'express': 'web_backend_and_apis',
    'spring-boot': 'web_backend_and_apis',
    'asp.net': 'web_backend_and_apis',
    'rest': 'web_backend_and_apis',
    'api': 'web_backend_and_apis',
    'graphql': 'web_backend_and_apis',
    # 4. Databases & Storage
    'sql': 'databases_and_storage',
    'mysql': 'databases_and_storage',
    'postgresql': 'databases_and_storage',
    'mongodb': 'databases_and_storage',
    'sqlite': 'databases_and_storage',
    'redis': 'databases_and_storage',
    'database': 'databases_and_storage',
    'nosql': 'databases_and_storage',
    # 5. Operating Systems & Systems
    'linux': 'operating_systems_and_systems',
    'unix': 'operating_systems_and_systems',
    'shell': 'operating_systems_and_systems',
    'bash': 'operating_systems_and_systems',
    'posix': 'operating_systems_and_systems',
    'memory-management': 'operating_systems_and_systems',
    'concurrency': 'operating_systems_and_systems',
    'multithreading': 'operating_systems_and_systems',
    # 6. Data Science & ML
    'pandas': 'data_science_and_ml',
    'numpy': 'data_science_and_ml',
    'dataframe': 'data_science_and_ml',
    'machine-learning': 'data_science_and_ml',
    'deep-learning': 'data_science_and_ml',
    'tensorflow': 'data_science_and_ml',
    'pytorch': 'data_science_and_ml',
    'scikit-learn': 'data_science_and_ml',
    # 7. Mobile App Development
    'android': 'mobile_development',
    'ios': 'mobile_development',
    'swift': 'mobile_development',
    'kotlin': 'mobile_development',
    'flutter': 'mobile_development',
    'react-native': 'mobile_development',
    'xcode': 'mobile_development',
    'android-studio': 'mobile_development',
    'xamarin': 'mobile_development',
    # 8. DevOps & Cloud Infrastructure
    'docker': 'devops_cloud_and_infrastructure',
    'kubernetes': 'devops_cloud_and_infrastructure',
    'amazon-web-services': 'devops_cloud_and_infrastructure',
    'azure': 'devops_cloud_and_infrastructure',
    'git': 'devops_cloud_and_infrastructure',
    'ci-cd': 'devops_cloud_and_infrastructure',
    # 9. Software Engineering & Testing
    'unit-testing': 'software_engineering_and_testing',
    'testing': 'software_engineering_and_testing',
    'pytest': 'software_engineering_and_testing',
    'junit': 'software_engineering_and_testing',
    'design-patterns': 'software_engineering_and_testing',
    # 10. Networks & Security
    'networking': 'computer_networks_and_security',
    'sockets': 'computer_networks_and_security',
    'security': 'computer_networks_and_security',
    'cryptography': 'computer_networks_and_security',
    'ssl': 'computer_networks_and_security',
    'tls': 'computer_networks_and_security',
    'authentication': 'computer_networks_and_security'
}

# Map question tags to topics
def map_topics(tags):
    return [anchor_tag_map[t] for t in tags if t in anchor_tag_map]

df['matched_topics'] = df['tag_list'].apply(map_topics)
df['num_matched_topics'] = df['matched_topics'].apply(lambda l: len(set(l)))

# Apply conflict resolution policy (dominant >= 0.60, else drop)
def resolve_conflict(matched):
    if not matched:
        return 'UNMAPPED'
    counts = Counter(matched)
    top_topic, top_count = counts.most_common(1)[0]
    if len(counts) == 1:
        return top_topic
    elif (top_count / len(matched)) >= 0.60:
        return top_topic
    else:
        return 'AMBIGUOUS_CONFLICT'

df['assigned_topic'] = df['matched_topics'].apply(resolve_conflict)

print('Mapping Status Breakdown:')
status_counts = df['assigned_topic'].value_counts()
for topic, cnt in status_counts.items():
    print(f'  {topic:<35} : {cnt:4d} ({cnt / len(df) * 100:5.1f}%)')

## 6. Exploratory Class Distribution & Taxonomy Evaluation

We isolate the cleanly mapped subset to compute the **exploratory class distribution** across the provisional categories.  
> **Disclaimer:** These values represent exploratory estimates on a bounded sample under provisional tag anchors, not finalized training set distributions.

In [ ]:
clean_df = df[~df['assigned_topic'].isin(['UNMAPPED', 'AMBIGUOUS_CONFLICT'])].copy()
total_clean = len(clean_df)
print(f'Cleanly mapped questions: {total_clean} / {len(df)} ({total_clean / len(df) * 100:.1f}%)\n')

class_counts = clean_df['assigned_topic'].value_counts()
class_dist = clean_df['assigned_topic'].value_counts(normalize=True) * 100

dist_table = pd.DataFrame({
    'Topic Category': class_counts.index,
    'Sample Count': class_counts.values,
    'Exploratory Share (%)': class_dist.values.round(1)
})
print(dist_table.to_string(index=False))

print('\nKey Observations on Category Representation:')
print('1. High Concentration: Mobile Development and Web Frontend account for over 50% of mapped questions.')
print('2. Low Representation: Algorithms & Data Structures (only ~1.1%) is severely under-represented in StackPulse.')
print('3. Unmapped Rate: Over 50% of questions lack domain-specific anchor tags because they are tagged only with language names (e.g., python, javascript, java).')

## 7. Strict Anti-Leakage Protocol & Conclusions

### Anti-Leakage Mandate
Tags are used strictly to synthesize supervisory ground-truth topic labels during data preparation.  
**Tags must NEVER be included as input features during model training.**  
The classifier's input features must be computed solely from cleaned natural question text (`title` + `body_text`).

### Key Conclusions for Member 1 Workstream
1. **Feasibility:** StackPulse can support defensible topic labels, but **only if** ambiguous multi-domain questions and pure generic language-syntax questions are systematically handled.
2. **Taxonomy Refinement Required:** A dedicated category for *Programming Fundamentals & Syntax* or hierarchical mapping rules are needed before finalizing `configs/topic_map.yaml`.
3. **Prerequisite to Baseline Training:** Faculty ratification of Decision D0 and formal sign-off on the expanded tag dictionary.